# TP2 — Asistente conversacional de planificación de viajes

**Inteligencia Artificial, UTN FRRo** — Lucio Cosentino, Joaquin Carlos Fernandez Da Silva, Aaron de Bernardo, Elias Danteo.

Este notebook es el entregable oficial de la cátedra. Importa y ejecuta el paquete `src/asistente_viajes/`; **no contiene lógica de negocio propia**, solo demostración y narración (ver `SKILL.md`, "Qué es el proyecto").

Requisito de aceptación: este notebook tiene que correr de arriba a abajo, con el kernel recién reiniciado, sobre una máquina limpia con `requirements.txt` instalado, `docker compose up -d` levantado y `.env` completo.

## 1. Caso de negocio y arquitectura

### El problema

Planificar un viaje implica juntar información de muchas fuentes (qué visitar, cuánto cuesta, si es seguro, cómo está el clima) y tomar decisiones con datos parciales, dados de a poco en una conversación natural. Este sistema actúa como un **asesor de viajes conversacional** para tres destinos piloto (Barcelona, Miami, Cancún): interpreta lo que el cliente cuenta en lenguaje libre, completa por preguntas lo que falta, arma un itinerario día a día con costo estimado, y responde consultas puntuales (dónde comer, si es seguro caminar de noche) citando solo información real recuperada de un corpus curado — nunca datos inventados por el modelo.

### Los cuatro pilares (agentes, RAG, embeddings, NLP con LLM)

- **Agentes:** el orquestador es un grafo de **LangGraph** (`src/asistente_viajes/grafo.py`) con nodos de responsabilidad única. Decide sola qué acción corresponde a cada turno (RF12), sin que el usuario indique un modo, y puede atender varios pedidos en el mismo mensaje.
- **RAG:** tres corpus (atractivos turísticos, comercios, FAQ del viajero) en **PostgreSQL + pgvector**, filtrados por destino y buscados semánticamente en una sola consulta SQL.
- **Embeddings:** `sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2` (384 dim), local — no gasta cuota de Gemini embebiendo los corpus.
- **NLP con LLM:** Gemini (`gemini-3.5-flash-lite`, con rotación de 3 claves) para extracción estructurada, decisión de acciones, y redacción de texto — siempre acotado al contexto recuperado, nunca inventando hechos.

### Flujo del orquestador

Cada turno de conversación pasa por este grafo (ver la celda de la sección 2 para el diagrama generado en vivo desde el código):

```
interpretar → actualizar_estado → planificar → ejecutar_acciones → disparar_info_destino → redactar
```

- **`interpretar`**: un único llamado estructurado al LLM que extrae los datos del viaje mencionados en el mensaje Y decide qué acciones pidió el cliente (puede ser más de una).
- **`actualizar_estado`**: lógica pura (nunca el LLM) — parsea fechas, fusiona el estado sin pisar lo ya cargado (RF2), detecta si el destino mencionado no es uno de los tres pilotos.
- **`planificar`**: lógica pura — decide qué ejecutar: si falta algún dato, una única pregunta consolidada con opciones concretas (no una por una); si ya hay destino, siempre se suma algo útil en el mismo turno, no se deja al cliente solo con la pregunta.
- **`ejecutar_acciones`**: corre cada acción pedida (armar el plan, recomendar actividades/locales, responder FAQ), cada una con su propio manejo de errores — una tool rota no tira abajo el turno completo.
- **`disparar_info_destino`**: única excepción a que decida el LLM (RF12) — clima e idioma/moneda se disparan solos al confirmarse el destino con fechas.
- **`redactar`**: si no se ejecutó ninguna acción (un saludo, un agradecimiento, una pregunta sobre algo ya hablado), un único llamado al LLM responde desde el historial de la conversación y el estado — no una respuesta enlatada.

### Por qué PostgreSQL + pgvector, no Chroma

Pedido explícito del profesor. El argumento técnico: **filtro por metadata (destino) y búsqueda semántica en una sola consulta SQL**, y las tablas estructuradas (itinerarios, conversaciones) viven en la misma base que los vectores — un solo motor en vez de dos. Ver `docs/DECISIONES.md`, D-01.

In [1]:
from asistente_viajes.grafo import construir_grafo

print(construir_grafo().get_graph().draw_mermaid())

---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	interpretar(interpretar)
	actualizar_estado(actualizar_estado)
	planificar(planificar)
	ejecutar_acciones(ejecutar_acciones)
	disparar_info_destino(disparar_info_destino)
	redactar(redactar)
	__end__([<p>__end__</p>]):::last
	__start__ --> interpretar;
	actualizar_estado --> planificar;
	disparar_info_destino --> redactar;
	ejecutar_acciones --> disparar_info_destino;
	interpretar --> actualizar_estado;
	planificar --> ejecutar_acciones;
	redactar --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc



## 2. Setup

Instalación (correr una sola vez, fuera del notebook):

```bash
python -m venv .venv && source .venv/bin/activate
pip install -r requirements.txt
cp .env.example .env   # completar las variables
docker compose up -d
python -m scripts.inicializar_db
```

Las celdas de abajo verifican que la configuración, la base de datos y el LLM están disponibles antes de seguir.

In [2]:
# asistente_viajes esta instalado como paquete (pip install -e .), pero
# scripts/ no: se agrega la raiz del repo al path para poder importar
# scripts.* mas abajo (seccion 3), igual que "python -m scripts.algo"
# desde la terminal.
import logging
import sys
from pathlib import Path

raiz_repo = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(raiz_repo) not in sys.path:
    sys.path.insert(0, str(raiz_repo))

# Librerias de terceros (httpx, huggingface_hub) en WARNING para no
# llenar el notebook de ruido; los logs propios (por ejemplo, la
# rotacion de claves de Gemini ante un 429/503/timeout, o los conteos
# de scripts.verificar_corpus) se dejan en INFO a proposito, son
# evidencia real de que las cosas funcionan.
logging.basicConfig(level=logging.WARNING, format="%(levelname)s %(message)s")
logging.getLogger("asistente_viajes").setLevel(logging.INFO)
logging.getLogger("scripts").setLevel(logging.INFO)

In [3]:
from asistente_viajes.config import cargar_configuracion

configuracion = cargar_configuracion()
print("LLM_PROVIDER:", configuracion.llm_provider)
print("GEMINI_MODEL:", configuracion.gemini_model)
print("cantidad de claves de Gemini:", len(configuracion.claves_gemini))
print("DATABASE_URL configurada:", bool(configuracion.database_url))

LLM_PROVIDER: gemini
GEMINI_MODEL: gemini-3.5-flash-lite
cantidad de claves de Gemini: 3
DATABASE_URL configurada: True


In [4]:
from asistente_viajes.db import obtener_conexion

with obtener_conexion() as conexion, conexion.cursor() as cursor:
    cursor.execute("SELECT count(*) FROM documento_rag;")
    (cantidad,) = cursor.fetchone()

print(f"conexion a Postgres OK, {cantidad} documentos en documento_rag")

conexion a Postgres OK, 108 documentos en documento_rag


In [5]:
from asistente_viajes.llm import contenido_texto, crear_rotador

rotador = crear_rotador()
respuesta = rotador.invocar("Respondé solo con la palabra: listo")
print("LLM OK, respuesta:", contenido_texto(respuesta))

INFO usando clave_1


WARNING Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


LLM OK, respuesta: listo


## 3. Ingesta y construcción de los corpus

El pipeline de ingesta (`src/asistente_viajes/ingesta/`) tiene tres pasos:

1. **`opentripmap.py`**: cliente de la API en dos pasos (búsqueda por radio, después detalle por `xid`), con extracto de Wikipedia cuando existe.
2. **`normalizar.py`**: filtra POIs sin texto descriptivo real (mínimo 200 caracteres — nunca se inventa texto para poder embeberlo), separa en corpus (`atractivos`/`comercios`) por `kind`, y excluye POIs cuyo kind primario es en realidad un edificio (torre de departamentos, hotel) y no un atractivo turístico (D-17: Miami tenía 49 de 75 "atractivos" que eran esto).
3. **`cargar_vectores.py`**: embebe con el modelo local y hace upsert en `documento_rag` por `xid` (idempotente).

`scripts/cargar_todos.py --reemplazar` corre el pipeline completo para los 3 destinos piloto. Ya se corrió contra las APIs reales para este demo (no se vuelve a correr acá para no repetir llamadas a OpenTripMap); la celda de abajo verifica el resultado real cargado en la base, con los conteos por destino/corpus/fuente.

In [6]:
from scripts.verificar_corpus import main as verificar_corpus

_codigo_salida = verificar_corpus()
assert _codigo_salida == 0, "algun destino piloto quedo por debajo del minimo de atractivos"
print("\ntodos los destinos piloto por encima del minimo")

INFO Barcelona  atractivos  opentripmap 39


INFO Barcelona  comercios   opentripmap 2


INFO Barcelona  faq         curado      3


INFO Cancun     atractivos  curado      20


INFO Cancun     faq         curado      4


...

INFO Miami      faq         curado      3


INFO Barcelona: 39 atractivos, OK (minimo 20)


INFO Cancun: 20 atractivos, OK (minimo 20)


INFO Miami: 35 atractivos, OK (minimo 20)



todos los destinos piloto por encima del minimo


## 4. Prueba de recuperación aislada

`recuperacion/_consulta.py` tiene la consulta canónica de los tres corpus: filtro por `corpus` y `destino` (metadata) más orden por similitud semántica (`embedding <=> consulta`), **las dos cosas en una sola consulta SQL** — el argumento concreto a favor de pgvector sobre Chroma.

Abajo, la misma consulta pero mostrando también la distancia coseno de cada resultado (no expuesta en el contrato normal de la tool, se arma acá con SQL directo solo para la demo).

In [7]:
from asistente_viajes.embeddings import embeber_texto

consulta_texto = "museos de historia maya"
destino_demo = "Cancun"
vector_consulta = embeber_texto(consulta_texto)

sql = "SELECT nombre, categoria, embedding <=> %(consulta)s::vector AS distancia FROM documento_rag WHERE corpus = 'atractivos' AND lower(unaccent(destino)) = lower(unaccent(%(destino)s)) ORDER BY distancia LIMIT 5;"

with obtener_conexion() as conexion, conexion.cursor() as cursor:
    cursor.execute(sql, {"consulta": vector_consulta, "destino": destino_demo})
    resultados = cursor.fetchall()

print(f"Consulta: {consulta_texto!r} en {destino_demo}\n")
for nombre, categoria, distancia in resultados:
    print(f"  {distancia:.4f}  [{categoria}]  {nombre}")

WARNING Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Consulta: 'museos de historia maya' en Cancun

  0.2199  [museums]  Museo Maya de Cancun
  0.3771  [historic]  Chichen Itza
  0.3939  [historic]  San Miguelito
  0.4332  [historic]  Coba (zona arqueologica)
  0.4347  [historic]  Yamil Lu'um (Templo del Alacran)


## 5. Slot filling paso a paso (RF1, RF2)

`tools/completar_slots.py` extrae los datos del viaje de un mensaje libre con salida estructurada del LLM, los fusiona **sin pisar lo ya cargado** (merge no destructivo, RF2), y arma una única pregunta consolidada por todo lo que falta (D-14) — determinística, con opciones concretas, sin pasar por el LLM.

In [8]:
from asistente_viajes.estado import PreferenciasViaje
from asistente_viajes.tools.completar_slots import completar_slots

estado = PreferenciasViaje()

estado, pregunta = completar_slots(
    rotador, "Quiero ir a Cancún, me interesa la playa e historia maya", estado
)
print("Turno 1 — estado:", estado.model_dump(exclude_none=True))
print("Turno 1 — pregunta:", pregunta)

INFO usando clave_2


Turno 1 — estado: {'destino': 'Cancun', 'tipo_destino': 'playa', 'intereses': ['playa', 'historia maya']}
Turno 1 — pregunta: ¿Qué presupuesto tiene en mente (bajo, medio, alto), le sugiero medio? ¿Para cuándo (fechas exactas o la cantidad de días), le sugiero 5 días, fecha a confirmar? ¿Cuántas personas viajan, le sugiero 2? Si prefiere, dígame "usar sugerencias" y armo un primer borrador con esos valores.


In [9]:
estado, pregunta = completar_slots(
    rotador, "Somos 2 personas, presupuesto medio, del 10 al 15 de enero de 2027", estado
)
print("Turno 2 — estado:", estado.model_dump(exclude_none=True))
print("Turno 2 — pregunta:", pregunta)  # None: ya está todo, no repregunta nada de lo ya cargado

INFO usando clave_3


Turno 2 — estado: {'destino': 'Cancun', 'tipo_destino': 'playa', 'intereses': ['playa', 'historia maya'], 'presupuesto': 'medio', 'fecha_inicio': datetime.date(2027, 1, 10), 'fecha_fin': datetime.date(2027, 1, 15), 'cantidad_personas': 2}
Turno 2 — pregunta: None


## 6. Conversación completa de punta a punta — caso de demo principal

`agente.procesar_mensaje` es la fachada del orquestador (RF11): mantiene `SesionAgente` (estado del viaje + historial) entre turnos, y en cada turno corre el grafo completo de la sección 1. El caso de abajo muestra, en una sola conversación real:

- **RF1/RF2** (slot filling) y **D-14** (pregunta consolidada con "usar sugerencias").
- **RF12** (el orquestador decide sola qué hacer, nunca el usuario indica un modo).
- **Multi-intent** (D-12): un mensaje con varios pedidos a la vez.
- **RF8** (`info_destino`, disparo automático al confirmarse destino y fechas).
- **Memoria de conversación** (D-12): una pregunta de recap contestada desde el historial, no una frase enlatada.

In [10]:
from asistente_viajes.agente import SesionAgente, procesar_mensaje

sesion = SesionAgente()


def turno(mensaje: str) -> None:
    with obtener_conexion() as conexion:
        respuesta = procesar_mensaje(conexion, rotador, sesion, mensaje)
    print(f"Cliente: {mensaje}\n")
    print(f"Asistente: {respuesta}\n")
    print("-" * 70)


turno("Hola, quiero ir a Barcelona")

INFO usando clave_1


INFO usando clave_2


Cliente: Hola, quiero ir a Barcelona

Asistente: ¿Qué le interesa hacer (historia, naturaleza, gastronomia, compras, descanso, vida nocturna)? ¿Qué presupuesto tiene en mente (bajo, medio, alto), le sugiero medio? ¿Para cuándo (fechas exactas o la cantidad de días), le sugiero 5 días, fecha a confirmar? ¿Cuántas personas viajan, le sugiero 2? Si prefiere, dígame "usar sugerencias" y armo un primer borrador con esos valores.

No encontré actividades para recomendarle con esos intereses en este destino.

----------------------------------------------------------------------


In [11]:
turno("usar sugerencias")

INFO usando clave_3


INFO usando clave_1


Cliente: usar sugerencias

Asistente: Con mucho gusto utilizaré las sugerencias predeterminadas para avanzar con su viaje a Barcelona. Aún no puedo planificar el viaje completo, ya que en esta fase piloto solo realizo estas coordinaciones iniciales, pero estoy a su disposición para seguir conversando.

----------------------------------------------------------------------


In [12]:
turno("armame el plan, y decime qué puedo visitar relacionado con arquitectura")

INFO usando clave_2


INFO usando clave_3


Cliente: armame el plan, y decime qué puedo visitar relacionado con arquitectura

Asistente: Armé un plan de 5 día(s) para **Barcelona**:
- **Día 1**: Banc Vitalici d'Espanya, Antic Banc Central, Casa Magarola (costo estimado $70)
- **Día 2**: The Goddess, House in carrer Tallers, 45, Allegory of Barcelona (costo estimado $70)
- **Día 3**: Casa Emilia Carles, Font ornamental del passeig de Gràcia, Casa Bosch Alsina (costo estimado $70)
- **Día 4**: Casa Pich i Pon, Palau Marcet, Casa Pia Batlló (costo estimado $70)
- **Día 5**: Casa Heribert Pons, Casa Pratjusà, Casa Jaime Moysi (costo estimado $70)

Costo total estimado: USD 350 por persona, USD 700 para el grupo de 2.
(sin fecha de inicio confirmada, el plan es por cantidad de dias)

- **Banc Vitalici d'Espanya**: Se trata de un edificio incluido en el Inventario del Patrimonio Arquitectónico de Cataluña con 21 plantas y cuatro cuerpos.
- **Casa Pich i Pon**: Es un edificio diseñado por el arquitecto Josep Puig i Cadafalch con pisos 

In [13]:
turno("¿a qué destino dijimos que quería ir, y con qué presupuesto?")

INFO usando clave_1


INFO usando clave_2


Cliente: ¿a qué destino dijimos que quería ir, y con qué presupuesto?

Asistente: Con gusto le recuerdo que su destino elegido es Barcelona y que estamos manejando un presupuesto medio para 2 personas. Quedo a su disposición para cualquier otra consulta sobre su viaje.

----------------------------------------------------------------------


## 7. Consultas puntuales de recomendación local (RF3, RF4, RF9)

Fuera del flujo de armar el plan completo, el cliente puede preguntar algo puntual en cualquier momento: actividades por interés, dónde comer/comprar, o seguridad y costumbres. Cada una recupera del corpus correspondiente y responde **solo** con lo que ese texto dice.

In [14]:
from asistente_viajes.tools.recomendar_actividades import recomendar_actividades

with obtener_conexion() as conexion:
    actividades = recomendar_actividades(
        conexion, rotador, destino="Cancun", intereses=["naturaleza"], k=3
    )

for actividad in actividades:
    print(f"- {actividad.nombre}: {actividad.justificacion}")

INFO usando clave_3


- Jardin Botanico Dr. Alfredo Barrera Marin: El lugar cuenta con 65 hectareas de selva protegida, secciones de plantas y puentes colgantes ideales para usted si busca naturaleza.
- Museo Subacuatico de Arte (MUSA): Se encuentra dentro del Parque Nacional Marino de Cancun y ofrece conservacion marina y habitats artificiales para el coral.
- Cenote Verde Lucero: Esta ubicado en la selva cercana a Puerto Morelos y cuenta con dos cenotes de aguas cristalinas ideales para usted.


In [15]:
from asistente_viajes.tools.responder_faq_viajero import responder_faq_viajero

with obtener_conexion() as conexion:
    respuesta_faq = responder_faq_viajero(
        conexion, rotador, "Barcelona", "¿cuánto se deja de propina?"
    )

print("Respondida:", respuesta_faq.respondida)
print("Respuesta:", respuesta_faq.respuesta)
print("Temas usados:", respuesta_faq.temas_usados)

INFO usando clave_1


Respondida: True
Respuesta: Le informo que la propina no es obligatoria en Espana. Si el servicio estuvo bien, alcanza con redondear la cuenta o dejar algunas monedas, ya que no se espera un porcentaje fijo como en otros paises.
Temas usados: ['Horarios de comida y propinas']


## 8. Extensiones implementadas

- **RF8, clima + idioma/moneda** (`tools/info_destino.py`): ya se vio disparado automáticamente en la sección 6. No es RAG — el clima es una llamada en vivo (gratis, sin API key), idioma/moneda salen de una tabla de referencia chica.
- **RF9, FAQ del viajero**: sección 7 de arriba.
- **RF6/RF7, alojamiento y vuelos** (`services/rapidapi/`, sobre Booking.com15 — Amadeus fue dado de baja, D-05/D-06): tools implementadas y testeadas, con fallback a datos de ejemplo marcados explícitamente si la API externa falla. Deliberadamente **no** enganchadas al orquestador todavía (quedan para una próxima iteración, ver `estado.md`), se muestran acá invocadas directo.
- **D-13, persistencia de conversaciones**: cada chat se puede guardar y recuperar completo (Postgres), con historial y estado. Ver la GUI (`ui/chat_app.py`, Fase 7B/7C) para la demo interactiva con varias conversaciones guardadas en simultáneo.

In [16]:
from datetime import date

from asistente_viajes.services.rapidapi.booking import buscar_alojamiento

with obtener_conexion() as conexion:
    alojamientos = buscar_alojamiento(conexion, "Barcelona", date(2027, 3, 5), date(2027, 3, 10))

for alojamiento in alojamientos[:3]:
    marca = " (dato de ejemplo)" if alojamiento.es_fixture else ""
    print(f"- {alojamiento.nombre}: {alojamiento.precio_total} {alojamiento.moneda}{marca}")

INFO booking api/v1/hotels/searchHotels -> 200 en 1657ms (intento 1)


- Hotel Best Aranea: 975.341759311711 USD
- H10 Casanova: 1107.54416403169 USD
- Motel One Barcelona-Ciutadella: 1174.5685217767 USD


In [17]:
from asistente_viajes.conversaciones import cargar_conversacion, crear_conversacion, guardar_turno

with obtener_conexion() as conexion:
    id_demo = crear_conversacion(conexion)
    guardar_turno(conexion, id_demo, sesion, "mensaje de ejemplo", "respuesta de ejemplo")
    conexion.commit()
    recargada = cargar_conversacion(conexion, id_demo)

print("conversacion guardada:", id_demo)
print("historial recuperado:", recargada.historial)

conversacion guardada: b0a41b41-8d89-45ac-ba8e-f51324882e39
historial recuperado: [{'rol': 'usuario', 'texto': 'mensaje de ejemplo'}, {'rol': 'asistente', 'texto': 'respuesta de ejemplo'}]


## 9. Casos de prueba y resultados

### 9.1 Caso positivo: destino fuera de alcance manejado con claridad

El sistema solo tiene datos reales de 3 destinos piloto. Pedir un destino distinto no debe perderse en silencio.

In [18]:
sesion_tokio = SesionAgente()
with obtener_conexion() as conexion:
    respuesta = procesar_mensaje(
        conexion, rotador, sesion_tokio, "Quiero ir a Tokio, me gusta la comida y la tecnología"
    )
print(respuesta)

INFO usando clave_2


Por ahora no tengo datos de "Tokio". Por ahora trabajo con estos destinos:
- **Barcelona**: Ciudad mediterranea con arquitectura modernista (Gaudi), historia y gastronomia.
- **Cancun**: Playas caribenas, cenotes y ruinas mayas cerca (Chichen Itza, Tulum, Coba).
- **Miami**: Playas, vida nocturna y una fuerte influencia latina (Little Havana, Calle Ocho).

¿A qué destino le gustaría viajar (Barcelona, Cancun, Miami)? ¿Qué presupuesto tiene en mente (bajo, medio, alto), le sugiero medio? ¿Para cuándo (fechas exactas o la cantidad de días), le sugiero 5 días, fecha a confirmar? ¿Cuántas personas viajan, le sugiero 2? Si prefiere, dígame "usar sugerencias" y armo un primer borrador con esos valores.


### 9.2 Caso límite documentado: RF4 (recomendación de comercios) sin volumen suficiente de datos

**Esto es una limitación real y conocida del sistema, no un bug** (ver D-07 en `docs/DECISIONES.md`). OpenTripMap no da volumen confiable de comercios/gastronomía sin curaduría manual de horas en ningún destino piloto, así que RF4 se movió de núcleo a extensión: el corpus de comercios existe pero está incompleto. El sistema responde honestamente que no encontró nada, **en vez de inventar un local que no está en los datos** — es exactamente el comportamiento correcto dado el principio de "nada inventado" (restricción dura 5), aunque la experiencia para el usuario sea limitada en este punto.

In [19]:
with obtener_conexion() as conexion:
    respuesta = procesar_mensaje(
        conexion, rotador, sesion, "¿dónde puedo comprar algo típico y barato?"
    )
print(respuesta)

INFO usando clave_3


INFO usando clave_1


No encontré locales para recomendarle con esa consulta en este destino.


### 9.3 Caso límite documentado: pronóstico de clima fuera de horizonte

Open-Meteo (la fuente de clima en vivo) solo pronostica ~16 días para adelante. Para fechas más lejanas, el sistema lo dice explícitamente en vez de mostrar un número inventado (ver `tools/info_destino.py`).

In [20]:
from datetime import UTC, datetime, timedelta

from asistente_viajes.tools.info_destino import obtener_clima

lejos = datetime.now(tz=UTC).date() + timedelta(days=200)
clima = obtener_clima(
    lat=41.3874, lon=2.1686, fecha_inicio=lejos, fecha_fin=lejos + timedelta(days=3)
)
print("disponible:", clima.disponible)
print("detalle:", clima.detalle)

disponible: False
detalle: Todavía no tengo pronóstico confiable para esas fechas (el clima en vivo solo llega hasta 16 días por delante); más cerca del viaje puedo consultarlo de nuevo.


### Resumen de dificultades reales encontradas

El detalle completo, con fecha, causa y solución de cada una, vive en `docs/DIFICULTADES.md` (insumo directo de la defensa oral y del video). Los puntos más relevantes:

- **P-06/P-08:** la latencia real de Gemini bajo uso acumulado (hasta 42s) resultó ser el SDK reintentando 6 veces por su cuenta antes de que el rotador de claves propio viera el error — no un problema del rotador en sí.
- **P-07:** el LLM de extracción llegó a copiar características de un destino inferido (dadas solo para identificar la ciudad) como si fueran gustos del usuario, violando la regla de "nada inventado"; se corrigió acotando explícitamente el prompt.
- **P-10:** un mensaje con varios pedidos a la vez diluía la búsqueda semántica de cada uno si se le pasaba el mensaje completo como consulta a cada tool; se resolvió con un fragmento de consulta por acción.
- **D-07:** el corpus de comercios (RF4) nunca alcanzó volumen suficiente por API sola, quedó como extensión con datos parciales — ver 9.2 arriba.